### Lab
load → split → embed → Chroma → RetrievalQA → Tool → agent.
Dataset :Shakespeare’s sonnets and Ruff

In [1]:
#Install dependencies

%pip install -q "langchain-classic>=1,<2" "langchain-openai>=1,<2" "langchain-community>=0.4,<0.5" "langchain-chroma>=0.2,<2" "langchain-text-splitters>=1,<2" beautifulsoup4 requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Initial Setup
from langchain_classic.chains import RetrievalQA
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from pathlib import Path
from dotenv import load_dotenv
import os
import sys

print("Python:", sys.version)
print("Working directory:", Path.cwd())
load_dotenv(Path.cwd() / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add OPENAI_API_KEY to .env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)



/tmp/ipykernel_42483/3431072829.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Working directory: /home/haripriya/ironhack/15_ai_engineering/lab-agent-vector-store


In [3]:
#Get the dataset- download the raw text manually to datasets/sonnets.txt.
import requests

Path("datasets").mkdir(exist_ok=True)
doc_path = Path("datasets/sonnets.txt")
url = "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"
if not doc_path.exists():
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    doc_path.write_text(response.text, encoding="utf-8")

print(doc_path.resolve())
print(doc_path.read_text(encoding="utf-8")[:500])


/home/haripriya/ironhack/15_ai_engineering/lab-agent-vector-store/datasets/sonnets.txt
	SONNETS



TO THE ONLY BEGETTER OF
THESE INSUING SONNETS
MR. W. H. ALL HAPPINESS
AND THAT ETERNITY
PROMISED BY
OUR EVER-LIVING POET WISHETH
THE WELL-WISHING
ADVENTURER IN
SETTING FORTH
T. T.


I.

FROM fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light'st flame with self-substantial fuel,
Making a famine where abundance l


In [4]:
#Load, split, embed, and build the vector store
import uuid
run_id = uuid.uuid4().hex[:8]

loader = TextLoader(str(doc_path), encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print("Documents:", len(documents))
print("Chunks:", len(texts))
print(texts[0].page_content[:500])

sonnets_db = Chroma.from_documents(
    texts, embeddings, collection_name=f"shakespeare-sonnets-{run_id}"
)


Documents: 1
Chunks: 154
SONNETS

TO THE ONLY BEGETTER OF
THESE INSUING SONNETS
MR. W. H. ALL HAPPINESS
AND THAT ETERNITY
PROMISED BY
OUR EVER-LIVING POET WISHETH
THE WELL-WISHING
ADVENTURER IN
SETTING FORTH
T. T.


I.

FROM fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light'st flame with self-substantial fuel,
Making a famine where abundance lies


In [5]:
#Create the new RetrievalQA chain

sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)

# Test the source directly before involving the agent.
print(sonnets.invoke({"query": "What comparison opens Sonnet 18?"})["result"])


Sonnet 18 opens with the comparison "Shall I compare thee to a summer's day?"


In [6]:
#Adding a second knowledge source

import bs4
os.environ.setdefault("USER_AGENT", "AgentVectorStoreLab/1.0")
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://docs.astral.sh/ruff/faq/",
    bs_kwargs={"parse_only": bs4.SoupStrainer("article")},
    requests_kwargs={"timeout": 60},
    raise_for_status=True,
)
docs = loader.load()
if not docs or not docs[0].page_content.strip():
    raise ValueError("No Ruff article text loaded; check the URL and HTML selector.")

ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(
    ruff_texts, embeddings, collection_name=f"ruff-{run_id}"
)
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)
print("Ruff chunks:", len(ruff_texts))
print(ruff.invoke({"query": "How does Ruff compare to Flake8?"})["result"])


Created a chunk of size 2210, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000


Ruff chunks: 19
Ruff can be used as a drop-in replacement for Flake8 when used without or with a small number of plugins, alongside Black, and on Python 3 code. Under those conditions, Ruff implements every rule in Flake8, including all of the F rules (which originate from Pyflakes) and a subset of the E and W rules (which originate from pycodestyle).

Ruff also re-implements some of the most popular Flake8 plugins and related code quality tools natively. Additionally, Ruff can replace other tools like Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade.

However, Ruff does not support custom lint rules, unlike Flake8, which allows for plugins (called "checkers"). Ruff's primary limitation compared to Flake8 is that it does not support third-party plugins, although a plugin system is within-scope for the project.


In [7]:
#Create the Agent
from langchain_classic.agents import AgentType, initialize_agent
from langchain_core.tools import Tool

tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets.run,
        description="Useful for answering questions about Shakespeare's sonnets, including Sonnet 18, imagery, love, beauty, and time. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="Useful for answering questions about Ruff, a Python linter and formatter, including Flake8 comparisons and documented capabilities. Input should be a fully formed question.",
    ),
]
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, return_intermediate_steps=True,
    handle_parsing_errors=True, max_iterations=6,
)


/tmp/ipykernel_42483/1651392600.py:17: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


In [8]:
#Test 1 — new dataset only

sonnets_result = agent.invoke({
    "input": "Use the sonnets source to explain the opening comparison in Sonnet 18."
})
print(sonnets_result["output"])
print("Tools used:", [a.tool for a, _ in sonnets_result["intermediate_steps"]])




> Entering new AgentExecutor chain...
I need to analyze the opening lines of Sonnet 18, which famously begins with a comparison between the beloved and a summer's day. This will involve looking at the imagery and themes presented in those lines. 
Action: Shakespeare Sonnets QA System
Action Input: "Explain the opening comparison in Sonnet 18."
Observation: In the opening of Sonnet 18, the speaker poses the question, "Shall I compare thee to a summer's day?" This comparison suggests that the subject of the poem is being likened to the beauty and warmth of a summer day. However, the speaker quickly asserts that the subject is "more lovely and more temperate" than a summer's day, indicating that the subject possesses a beauty that is more consistent and enduring than the fleeting nature of summer. The comparison sets the tone for the rest of the sonnet, where the speaker explores the theme of eternal beauty and the power of poetry to immortalize that beauty.
Thought:I now know the final

In [9]:
#Test 2 — Ruff only
ruff_result = agent.invoke({
    "input": "Use the Ruff documentation to explain why someone might use Ruff over Flake8."
})
print(ruff_result["output"])
print("Tools used:", [a.tool for a, _ in ruff_result["intermediate_steps"]])




> Entering new AgentExecutor chain...
I need to gather information about Ruff and its advantages over Flake8 from the Ruff documentation. This will help me provide a clear comparison and reasoning for choosing Ruff.  
Action: Ruff QA System  
Action Input: Why might someone choose Ruff over Flake8?  
Observation: Someone might choose Ruff over Flake8 for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which can reduce the need for multiple plugins.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **Compatibility with Black**: Ruff is compatible with Black out-of-the-box, making it easier to integrate into projects that use Black for formatting.

5. **No Custom Rules**: Ruff does not support custom lint ru

In [10]:
#Multi-Hop vector store reasoning

multi_hop_question = (
    "First use the Shakespeare Sonnets QA System to identify the opening "
    "comparison in Sonnet 18. Then use the Ruff QA System, including that "
    "comparison in your question, to ask whether Ruff's documented capabilities "
    "include judging the literary quality of that text when stored in a Python "
    "string. Report the comparison and the relevant Ruff capabilities. "
    "Distinguish what the documentation states from your inference, and say "
    "when the retrieved context is insufficient."
)
multi_result = agent.invoke({"input": multi_hop_question})
print(multi_result["output"])

for action, observation in multi_result["intermediate_steps"]:
    print("\nTool:", action.tool)
    print("Input:", action.tool_input)
    print("Observation:", observation)




> Entering new AgentExecutor chain...
I will first use the Shakespeare Sonnets QA System to identify the opening comparison in Sonnet 18. 

Action: Shakespeare Sonnets QA System  
Action Input: "What is the opening comparison in Shakespeare's Sonnet 18?"

Observation: The opening comparison in Shakespeare's Sonnet 18 is between the beloved and a summer's day. The speaker asks, "Shall I compare thee to a summer's day?" and then states that the beloved is "more lovely and more temperate" than a summer's day.
Thought:I now have the opening comparison from Sonnet 18. Next, I will use the Ruff QA System to inquire about its capabilities regarding judging the literary quality of this comparison when stored in a Python string.

Action: Ruff QA System  
Action Input: "Does Ruff's documented capabilities include judging the literary quality of the text 'Shall I compare thee to a summer's day? Thou art more lovely and more temperate' when stored in a Python string?"

Observation: I don't know.

In [11]:
#Use the Agent solely as a router
router_tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets.run,
        description="Useful for answering questions about Shakespeare's sonnets, including Sonnet 18, imagery, love, beauty, and time. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="Useful for answering questions about Ruff, a Python linter and formatter, including Flake8 comparisons and documented capabilities. Input should be a fully formed question.",
        return_direct=True,
    ),
]
router = initialize_agent(
    router_tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, return_intermediate_steps=True,
    handle_parsing_errors=True, max_iterations=6,
)


In [12]:
router_sonnets = router.invoke({
    "input": "Use the sonnets source to explain the opening comparison in Sonnet 18."
})
print(router_sonnets["output"])
print("Tools used:", [a.tool for a, _ in router_sonnets["intermediate_steps"]])




> Entering new AgentExecutor chain...
I need to analyze the opening lines of Sonnet 18, which famously begins with a comparison between the beloved and a summer's day. This will involve looking at the imagery and themes presented in those lines. 
Action: Shakespeare Sonnets QA System
Action Input: "Explain the opening comparison in Sonnet 18."
Observation: In the opening of Sonnet 18, the speaker compares the beloved to a summer's day. This comparison highlights the beloved's beauty and loveliness, suggesting that they are even more beautiful and temperate than a summer's day. The speaker notes that summer can be marred by rough winds and has a limited duration, implying that while summer is beautiful, it is also fleeting and subject to imperfections. By contrasting the beloved with a summer's day, the speaker emphasizes the enduring and superior nature of the beloved's beauty, which will not fade or diminish over time.


> Finished chain.
In the opening of Sonnet 18, the speaker com

In [13]:
router_ruff = router.invoke({
    "input": "Use the Ruff documentation to explain why someone might use Ruff over Flake8."
})
print(router_ruff["output"])
print("Tools used:", [a.tool for a, _ in router_ruff["intermediate_steps"]])




> Entering new AgentExecutor chain...
I need to gather information about Ruff and its advantages over Flake8 from the Ruff documentation. This will help me provide a clear comparison and reasoning for choosing Ruff.  
Action: Ruff QA System  
Action Input: Why might someone choose to use Ruff over Flake8?  
Observation: Someone might choose to use Ruff over Flake8 for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which can cover a broader range of linting issues.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, which simplifies the setup and usage for users who prefer a more standardized approach.

5. **Compatibility with Black**: Ruff's lint

In [14]:
router_multi = router.invoke({"input": multi_hop_question})
print(router_multi["output"])

for action, observation in router_multi["intermediate_steps"]:
    print("\nTool:", action.tool)
    print("Input:", action.tool_input)
    print("Observation:", observation)




> Entering new AgentExecutor chain...
I will first identify the opening comparison in Shakespeare's Sonnet 18 using the Shakespeare Sonnets QA System. 

Action: Shakespeare Sonnets QA System  
Action Input: "What is the opening comparison in Sonnet 18?"  

Observation: The opening comparison in Sonnet 18 is "Shall I compare thee to a summer's day?" The speaker is contemplating whether to compare the person they are addressing to the beauty of a summer day.


> Finished chain.
The opening comparison in Sonnet 18 is "Shall I compare thee to a summer's day?" The speaker is contemplating whether to compare the person they are addressing to the beauty of a summer day.

Tool: Shakespeare Sonnets QA System
Input: What is the opening comparison in Sonnet 18?"  

Observation: The opening comparison in Sonnet 18 is "Shall I compare thee to a summer's day?" The speaker is contemplating whether to compare the person they are addressing to the beauty of a summer day.


In [15]:
#Inspect retrieved evidence

for label, store, question in [
    ("Sonnets", sonnets_db, "What comparison opens Sonnet 18?"),
    ("Ruff", ruff_db, "What does Ruff check in Python code?"),
]:
    print("\nSOURCE:", label)
    for document in store.as_retriever().invoke(question):
        print(document.metadata)
        print(document.page_content)
        print("---")



SOURCE: Sonnets
{'source': 'datasets/sonnets.txt'}
So is it not with me as with that Muse
Stirr'd by a painted beauty to his verse,
Who heaven itself for ornament doth use
And every fair with his fair doth rehearse
Making a couplement of proud compare,
With sun and moon, with earth and sea's rich gems,
With April's first-born flowers, and all things rare
That heaven's air in this huge rondure hems.
O' let me, true in love, but truly write,
And then believe me, my love is as fair
As any mother's child, though not so bright
As those gold candles fix'd in heaven's air:
  Let them say more than like of hearsay well;
  I will not praise that purpose not to sell.

XXII.
---
{'source': 'datasets/sonnets.txt'}
SONNETS

TO THE ONLY BEGETTER OF
THESE INSUING SONNETS
MR. W. H. ALL HAPPINESS
AND THAT ETERNITY
PROMISED BY
OUR EVER-LIVING POET WISHETH
THE WELL-WISHING
ADVENTURER IN
SETTING FORTH
T. T.


I.

FROM fairest creatures we desire increase,
That thereby beauty's rose might never die,
But a

## Reflection

### Tool selection
For the Sonnet 18 question, the agent selected the Shakespeare Sonnets
QA System. It explained the comparison between the beloved and a
summer’s day.

For the Ruff-versus-Flake8 question, the agent selected the Ruff QA
System. The response discussed performance, linting rules, automatic
fixes, and compatibility with Black.

Both individual questions were routed to the appropriate tool.

### Direct-return behavior
In the displayed executions, the agent finished immediately after the
tool returned its answer. This is consistent with return_direct=True:
the tool’s answer became the final response without further synthesis.

For the combined question, the displayed execution called only the
sonnets tool. It answered the literary part but did not consult Ruff,
so it did not complete both stages.

### Answer quality
Correct tool selection did not guarantee a fully reliable answer.
The Ruff answer presented the lack of custom rules as an advantage,
although this is a limitation for users who need custom checks.
Installing without Rust also does not, by itself, establish an
advantage over Flake8.

The Sonnet 18 explanation should clarify that the poem presents verse
as preserving the beloved’s beauty, rather than suggesting that the
person literally never ages.

### Evidence and limitations
The logs show tool calls and generated tool answers, not the raw
retrieved passages. Those passages must be inspected separately before
concluding that every claim is supported.

A normal-agent multi-hop trace is still needed to verify that both
tools were called and that the second request used the first result.